# 01 – SB 79 Capacity Engine: Berkeley Pilot
**Group 27 · CYPLAN 101 · MTC Project 3**

Midterm check-in goal: show the parcel-level pipeline working for one city.

1. Load MTC's SB 79 TOD zones (tier + distance band)
2. Load Alameda County parcels and keep Berkeley
3. Assign each parcel its SB 79 tier/band (spatial join)
4. Estimate allowed units = parcel acres × SB 79 density
5. Map + summary tables

**Data snapshot date:** 2026-10-21. Densities are SB 79 standards (Gov. Code §65912.155–.162); 200 ft values include the +40 du/ac adjacency bonus. Results are gross capacity (before subtracting existing units or ineligible parcels

In [45]:
# Run this first (Colab already has most of these)
!pip install -q geopandas folium mapclassify

In [46]:
import re
import geopandas as gpd
import pandas as pd
import folium

CRS_METERS = 26910          # UTM zone 10N (meters) for accurate areas
SQM_PER_ACRE = 4046.86

# SB 79 density standards (dwelling units / acre)
# Source: SB 79 (2025); ABAG SB 79 Summary (Apr 2026). 200 ft = adjacency (+40 du/ac).
DENSITY = {
    ("Tier 1", "200ft"):   160,
    ("Tier 1", "Quarter"): 120,
    ("Tier 1", "Half"):    100,
    ("Tier 2", "200ft"):   140,
    ("Tier 2", "Quarter"): 100,
    ("Tier 2", "Half"):     80,
}

## Step 1 – Load SB 79 TOD zones
In Colab, click the 📁 folder icon on the left and upload the TOD-zones file you downloaded from MTC's SB 79 map (GeoJSON, or a zipped shapefile). Put its filename below.

In [47]:
TOD_FILE = "/content/mtc_sb79_tod_zones_-1757785326437500512.geojson"    # <- change to your file name (a .zip shapefile also works)

tod = gpd.read_file(TOD_FILE)
print(len(tod), "zone polygons | CRS:", tod.crs)
print("Columns:", list(tod.columns))
tod.head()

665 zone polygons | CRS: EPSG:4326
Columns: ['OBJECTID', 'zone_id', 'zone_label', 'geometry']


,OBJECTID,zone_id,zone_label,geometry
0,1,764ca928-8bd7-4c6e-83d2-835dca264eed,Tier 1 - 200ft,"POLYGON Z ((-122.41005 37.62829 0, -122.41011 ..."
1,2,79f5642a-6435-4716-a694-03d275e109bf,Tier 1 - 200ft,"POLYGON Z ((-122.41571 37.63717 0, -122.41577 ..."
2,3,823ebbcc-0305-4986-ac7e-deb4954d6e39,Tier 1 - 200ft,"POLYGON Z ((-122.44385 37.66357 0, -122.44392 ..."
3,4,c60319c0-2b8b-4645-8fd4-961eda2fdf6b,Tier 1 - 200ft,"POLYGON Z ((-122.46749 37.68288 0, -122.46755 ..."
4,5,1e3f5c82-2986-4b56-8776-6d69a7fef2c3,Tier 1 - 200ft,"POLYGON Z ((-122.46866 37.70547 0, -122.46872 ..."


In [48]:
# Figure out tier + distance band from whatever the attribute columns are called.
# MTC labels look like "Tier 1 - Quarter Mile", "Tier 2 - 200ft", etc.
def parse_label(row):
    text = " ".join(str(v) for k, v in row.items() if k != "geometry").lower()
    tier = "Tier 1" if re.search(r"tier\s*_?1", text) else ("Tier 2" if re.search(r"tier\s*_?2", text) else None)
    if "200" in text:
        band = "200ft"
    elif "quarter" in text or "0.25" in text or "1/4" in text:
        band = "Quarter"
    elif "half" in text or "0.5" in text or "1/2" in text:
        band = "Half"
    else:
        band = None
    return pd.Series({"tier": tier, "band": band})

tod[["tier", "band"]] = tod.apply(parse_label, axis=1)
tod["density_du_ac"] = [DENSITY.get((t, b)) for t, b in zip(tod.tier, tod.band)]
print(tod.groupby(["tier", "band"], dropna=False).size())
print("Unparsed zones:", tod.density_du_ac.isna().sum(), "(check these columns by hand if > 0)")

tier    band   
Tier 1  200ft       73
        Half        54
        Quarter     58
Tier 2  200ft      339
        Half        59
        Quarter     82
dtype: int64
Unparsed zones: 0 (check these columns by hand if > 0)


## Step 2 – Load parcels and keep Berkeley

In [49]:
PARCEL_FILE = "/content/Parcels_7981198686543604007.zip"   # <- from data.acgov.org (Alameda County Parcel Boundaries)

# Berkeley bounding box (lon/lat). Swap for an official city boundary later.
BERKELEY_BBOX = (-13617157, 4557553, -13607027, 4566155)

parcels = gpd.read_file(PARCEL_FILE, bbox=BERKELEY_BBOX)
print(len(parcels), "parcels in Berkeley box | CRS:", parcels.crs)
print("Columns:", list(parcels.columns))

40579 parcels in Berkeley box | CRS: EPSG:3857
Columns: ['APN', 'APN_SORT', 'BOOK', 'PAGE', 'PARCEL', 'SUB_PARCEL', 'CLCA_CATEG', 'CENTROID_X', 'CENTROID_Y', 'SortParcel', 'PrintParce', 'TRAPrimary', 'TRASeconda', 'SitusStree', 'SitusStr_1', 'SitusUnit', 'SitusCity', 'SitusZip', 'SitusZip_4', 'Land', 'Imps', 'CLCALand', 'CLCAImps', 'HOEX', 'OTEX', 'TotalNetVa', 'LatestDocu', 'LatestDo_1', 'LatestDo_2', 'MailingAdd', 'MailingA_1', 'MailingA_2', 'MailingA_3', 'MailingA_4', 'UseCode', 'EconUnit', 'EconomicUn', 'SitusAddre', 'MailingA_5', 'geometry']


In [50]:
# Reproject both layers to meters so areas are correct
parcels = parcels.to_crs(CRS_METERS)
tod = tod.to_crs(CRS_METERS)

parcels["acres"] = parcels.geometry.area / SQM_PER_ACRE

# Find an APN-like column for IDs
apn_col = next((col for col in parcels.columns if "apn" in col.lower()), None)
if apn_col is None:
    parcels["parcel_id"] = range(len(parcels)); apn_col = "parcel_id"
print("Using ID column:", apn_col)
parcels[[apn_col, "acres"]].describe()

Using ID column: APN


,acres
count,40579.000000
mean,0.623065
std,23.358538
min,0.000051
25%,0.090936
50%,0.118012
75%,0.158810
max,4383.862628


## Step 3 – Spatial join: which SB 79 zone is each parcel in?
We use the parcel **centroid** (one assignment per parcel). If a centroid falls in overlapping zones, we keep the **highest** density standard that applies.

In [51]:
cent = parcels[[apn_col, "acres", "geometry"]].copy()
cent["geometry"] = cent.geometry.representative_point()   # always inside the parcel

joined = gpd.sjoin(cent, tod[["tier", "band", "density_du_ac", "geometry"]],
                   how="inner", predicate="within")

# keep best standard per parcel
best = (joined.sort_values("density_du_ac", ascending=False)
              .drop_duplicates(subset=apn_col)
              [[apn_col, "tier", "band", "density_du_ac"]])

tod_parcels = parcels.merge(best, on=apn_col, how="inner")
print(f"{len(tod_parcels):,} of {len(parcels):,} Berkeley parcels fall inside an SB 79 TOD zone")

11,347 of 40,579 Berkeley parcels fall inside an SB 79 TOD zone


## Step 4 – Estimate SB 79 capacity

In [52]:
tod_parcels["sb79_units"] = (tod_parcels["acres"] * tod_parcels["density_du_ac"]).round(1)

# Data-quality flags (good to report at check-in)
tiny = (tod_parcels.acres < 0.03).sum()   # < ~1,300 sq ft, likely slivers/condos
print("Parcels < 0.03 acres (possible slivers/condo pads):", tiny)

summary = (tod_parcels.groupby(["tier", "band"])
           .agg(parcels=(apn_col, "count"),
                acres=("acres", "sum"),
                sb79_units=("sb79_units", "sum"))
           .round(0))
summary.loc["TOTAL"] = summary.sum()
summary

Parcels < 0.03 acres (possible slivers/condo pads): 57


,parcels,acres,sb79_units
"(Tier 1, 200ft)",29.0,16.0,2496.0
"(Tier 1, Half)",7395.0,1314.0,131389.0
"(Tier 1, Quarter)",2277.0,355.0,42592.0
"(Tier 2, 200ft)",46.0,14.0,1893.0
"(Tier 2, Half)",1314.0,680.0,54407.0
"(Tier 2, Quarter)",286.0,77.0,7663.0
TOTAL,11347.0,2456.0,240440.0


> **Caveat for the check-in:** this is *gross theoretical* capacity. It doesn't yet subtract existing units, exclude non-residential/commercial-ineligible zoning, public land, historic sites, or local exemptions. Those are the next layers of the engine.

## Step 5 – Map

In [ ]:
m_parcels = tod_parcels[~tod_parcels.geometry.is_empty].to_crs(4326)
m_tod = tod[tod.tier.notna()].to_crs(4326)

m = folium.Map(location=[37.871, -122.273], zoom_start=14, tiles=None)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Basemap"
).add_to(m)

folium.GeoJson(
    m_tod[["tier", "band", "geometry"]],
    name="SB 79 TOD zones",
    style_function=lambda f: {"fillOpacity": 0, "color": "#555", "weight": 1, "dashArray": "4"},
).add_to(m)

m_parcels.explore(
    m=m, column="sb79_units", scheme="quantiles", k=5, cmap="viridis",
    legend=True, name="SB 79 allowed units per parcel",
    tooltip=[apn_col, "tier", "band", "acres", "sb79_units"],
    style_kwds={"weight": 0.2},
)
folium.LayerControl().add_to(m)
m

In [54]:
# Save outputs (download these from the Colab file panel and upload to GitHub)
m.save("berkeley_sb79_capacity_map.html")
summary.to_csv("berkeley_sb79_summary.csv")
print("Saved map + summary")

Saved map + summary


In [55]:
small = m_parcels[[apn_col, "tier", "band", "sb79_units", "geometry"]].copy()
small["geometry"] = small.to_crs(26910).geometry.simplify(3).to_crs(4326)
small["sb79_units"] = small["sb79_units"].round(0)

m2 = folium.Map(location=[37.871, -122.273], zoom_start=14, tiles=None)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri").add_to(m2)
small.explore(m=m2, column="sb79_units", scheme="quantiles", k=5, cmap="viridis",
              legend=True, tooltip=[apn_col, "tier", "band", "sb79_units"],
              style_kwds={"weight": 0.2})
m2.save("berkeley_sb79_map_small.html")
import os; print(round(os.path.getsize("berkeley_sb79_map_small.html")/1e6, 1), "MB")

6.0 MB


## Next steps (after check-in)
- Replace bbox with official Berkeley city boundary
- Subtract existing units (assessor data) → **net new** capacity
- Overlay MTC Equity Priority Communities
- Repeat for Palo Alto / a Santa Clara city
- GTFS: weekday departures per TOD stop (transit-quality index)